# Build Mamba-SSM wheel for CUDA 12.2 + PTX

This notebook clones the public repo, installs the build dependencies, and builds a wheel with PTX fallback enabled through `TORCH_CUDA_ARCH_LIST`.

It assumes a Colab GPU runtime with a CUDA 12.2-capable PyTorch install and builds a single `+PTX` wheel target.

In [ ]:
import os
import subprocess

REPO_URL = os.environ.get("REPO_URL", "https://github.com/davidkny22/efficient-mamba-ssm.git")
REPO_DIR = "/content/mamba"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repo:", os.getcwd())

In [ ]:
import glob as _glob
import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

def _pip(*args):
    return subprocess.call([sys.executable, "-m", "pip", "install", "--upgrade", *args])

BUILD_FROM_SOURCE = True

# mamba-ssm: grab pre-built wheel from Drive root if it already exists.
_mamba_wheel = _glob.glob("/content/drive/MyDrive/mamba_ssm*.whl")
if _mamba_wheel:
    _mamba_wheel.sort(reverse=True)
    print(f"Installing mamba-ssm from Drive: {_mamba_wheel[0]}")
    _pip(_mamba_wheel[0])
    BUILD_FROM_SOURCE = False
else:
    print("No mamba-ssm wheel on Drive root, building from source (slow)...")
    os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "12.2+PTX"
    _pip("mamba-ssm @ git+https://github.com/state-spaces/mamba.git", "--no-build-isolation") or print("mamba-ssm build failed")

# Patch kernels-community ssd_combined if it loads.
def _fix_ssd_combined_modules():
    try:
        import causal_conv1d_cuda as _cc
    except ImportError:
        return
    _fixed = 0
    for mod_name, mod in list(sys.modules.items()):
        if "ssd_combined" in mod_name and hasattr(mod, "causal_conv1d_cuda"):
            if getattr(mod, "causal_conv1d_cuda", None) is None:
                mod.causal_conv1d_cuda = _cc
                _fixed += 1
    if _fixed:
        print(f"Patched { _fixed } ssd_combined module(s) with causal_conv1d_cuda")

_fix_ssd_combined_modules()


In [ ]:
!python -m pip install --upgrade pip setuptools wheel ninja packaging
!python - <<'PY'
import torch
print('torch:', torch.__version__)
print('torch cuda:', torch.version.cuda)
PY

In [ ]:
import os

if BUILD_FROM_SOURCE:
    # Force a source build and add PTX fallback for Ada/Hopper-class cards.
    os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
    os.environ["MAMBA_FORCE_CXX11_ABI"] = "FALSE"
    os.environ["MAMBA_LOCAL_VERSION"] = "cu122ptx"
    os.environ["MAX_JOBS"] = "2"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "12.2+PTX"

    subprocess.run([sys.executable, "setup.py", "--name"], check=True)
    subprocess.run([sys.executable, "setup.py", "bdist_wheel", "--dist-dir", "dist"], check=True)
    subprocess.run(["bash", "-lc", "ls -lh dist"], check=True)
else:
    print("Pre-built wheel was installed from Drive; skipping source build.")

In [ ]:
import glob
print(glob.glob('dist/*.whl'))